# Donut Algorithm Zernike Comparison (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-03-24  
**Last Modified:** 2026-03-24  
**Status:** Draft  
**Keywords:** AOS, Donut, Wavefront, Zernike, Algorithm Comparison  

## Description

Compare Zernike wavefront coefficients between two different donut wavefront retrieval algorithms.

Key functionality:
1. Load two parquet files produced by `intrinsics_mktable.ipynb` for different algorithms
2. Match donuts between the two datasets using (thx_OCS, thy_OCS, day_obs, seq_num)
3. Filter for matched intra/extra focal pairs in both datasets
4. Make hexbin density plots comparing Zernike coefficients between algorithms

**Output:** Hexbin density plots and difference histograms comparing per-donut Zernike values between two algorithms

**Based on:** `intrinsics_mktable.ipynb`, `intrinsics_plots.ipynb`

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-03-24 | Aaron Roodman | Initial version |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Load Data](#load)
5. [Match Donuts](#match)
6. [Hexbin Density Plots](#scatter)

<a id='params'></a>
## Parameters

In [ ]:
from pathlib import Path

# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# Coordinate system (OCS throughout)
coord_sys = 'OCS'

# --- Algorithm A ---
prefix_A = 'fam_danish'
wep_ver_A = 'wep_v16_8_0'
dviz_ver_A = 'dviz_v3_5_0'

# --- Algorithm B ---
prefix_B = 'fam_danish_bin1'
wep_ver_B = 'wep_v16_8_0'
dviz_ver_B = 'dviz_v3_5_0'

# Parquet file date range (must match mktable output)
day_obs_min = 20260315
day_obs_max = 20260317

# Short labels for plots
label_A = f'{prefix_A}'
label_B = f'{prefix_B}'

# Output directory
output_dir = 'output'
Path(output_dir).mkdir(parents=True, exist_ok=True)

# Construct parquet file paths
file_A = f'{output_dir}/{prefix_A}_zernikes_{wep_ver_A}_{dviz_ver_A}_{day_obs_min}_{day_obs_max}.parquet'
file_B = f'{output_dir}/{prefix_B}_zernikes_{wep_ver_B}_{dviz_ver_B}_{day_obs_min}_{day_obs_max}.parquet'
visit_file_A = f'{output_dir}/{prefix_A}_zernikes_{wep_ver_A}_{dviz_ver_A}_{day_obs_min}_{day_obs_max}_visits.parquet'
visit_file_B = f'{output_dir}/{prefix_B}_zernikes_{wep_ver_B}_{dviz_ver_B}_{day_obs_min}_{day_obs_max}_visits.parquet'

print(f"Algorithm A: {label_A}")
print(f"  File: {file_A}")
print(f"Algorithm B: {label_B}")
print(f"  File: {file_B}")
print(f"Coordinate system: {coord_sys}")
print(f"day_obs range: {day_obs_min} - {day_obs_max}")

<a id='setup'></a>
## Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path

from astropy.table import QTable

import sys
sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting

setup_plotting()

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

<a id='functions'></a>
## Helper Functions

In [ ]:
def get_noll_indices(visit_info, nZk):
    """Get Noll Zernike indices from visit_info or fall back to contiguous.

    Parameters
    ----------
    visit_info : QTable or None
        Visit info table with optional nollIndices column.
    nZk : int
        Number of Zernike terms per donut.

    Returns
    -------
    iZs : list of int
        Noll indices.
    iZidx : dict
        Mapping from Noll index to column index.
    """
    if visit_info is not None and 'nollIndices' in visit_info.colnames:
        noll_arr = np.array(visit_info['nollIndices'][0])
        iZs = [int(n) for n in noll_arr]
        if len(iZs) != nZk:
            print(f"WARNING: nollIndices length ({len(iZs)}) != zk width ({nZk}), falling back")
            iZs = list(range(4, 4 + nZk))
    else:
        iZs = list(range(4, 4 + nZk))
        print(f"Warning: nollIndices not in visit_info — assuming contiguous Z4-Z{iZs[-1]}")
    iZidx = {iZ: i for i, iZ in enumerate(iZs)}
    return iZs, iZidx

<a id='load'></a>
## Load Data

In [ ]:
# Load Algorithm A
if not Path(file_A).exists():
    raise FileNotFoundError(f"File not found: {file_A}\nRun intrinsics_mktable.ipynb first.")
table_A = QTable.read(file_A)
print(f"Algorithm A ({label_A}): {len(table_A)} rows")

visit_info_A = None
if Path(visit_file_A).exists():
    visit_info_A = QTable.read(visit_file_A)
    print(f"  visit_info: {len(visit_info_A)} visits")

# Load Algorithm B
if not Path(file_B).exists():
    raise FileNotFoundError(f"File not found: {file_B}\nRun intrinsics_mktable.ipynb first.")
table_B = QTable.read(file_B)
print(f"\nAlgorithm B ({label_B}): {len(table_B)} rows")

visit_info_B = None
if Path(visit_file_B).exists():
    visit_info_B = QTable.read(visit_file_B)
    print(f"  visit_info: {len(visit_info_B)} visits")

In [ ]:
# Determine Noll indices for each dataset
zk_col = f'zk_{coord_sys}'
thx_col = f'thx_{coord_sys}'
thy_col = f'thy_{coord_sys}'

nZk_A = np.stack(table_A[zk_col]).shape[1]
nZk_B = np.stack(table_B[zk_col]).shape[1]

iZs_A, iZidx_A = get_noll_indices(visit_info_A, nZk_A)
iZs_B, iZidx_B = get_noll_indices(visit_info_B, nZk_B)

# Find common Zernike indices
iZs_common = [iZ for iZ in iZs_A if iZ in iZidx_B]
print(f"\nAlgorithm A Noll indices ({nZk_A} terms): {iZs_A}")
print(f"Algorithm B Noll indices ({nZk_B} terms): {iZs_B}")
print(f"Common Noll indices ({len(iZs_common)} terms): {iZs_common}")

<a id='match'></a>
## Match Donuts

Match donuts between the two algorithm runs using (thx_OCS, thy_OCS, day_obs, seq_num).
Only use donuts with matched intra/extra focal positions in both datasets.

In [ ]:
# Filter for matched intra/extra in both datasets
mask_matched_A = np.array(table_A['matched_intra_extra'])
mask_matched_B = np.array(table_B['matched_intra_extra'])

tA = table_A[mask_matched_A]
tB = table_B[mask_matched_B]
print(f"After matched_intra_extra filter:")
print(f"  Algorithm A: {len(tA)} donuts (from {len(table_A)})")
print(f"  Algorithm B: {len(tB)} donuts (from {len(table_B)})")

# Build matching keys: (day_obs, seq_num, thx_OCS, thy_OCS)
# Use rounded thx/thy for matching (to handle floating point)
def make_match_keys(table):
    """Create match keys from (day_obs, seq_num, thx_OCS, thy_OCS)."""
    day_obs = np.array(table['day_obs'])
    seq_num = np.array(table['seq_num'])
    thx = np.array(table[thx_col])
    thy = np.array(table[thy_col])
    # Round to 8 decimal places (sub-arcsecond precision) to avoid float mismatch
    keys = list(zip(day_obs, seq_num, np.round(thx, 8), np.round(thy, 8)))
    return keys

keys_A = make_match_keys(tA)
keys_B = make_match_keys(tB)

# Build index for B
key_to_idx_B = {}
for i, key in enumerate(keys_B):
    key_to_idx_B[key] = i

# Match A to B
matched_idx_A = []
matched_idx_B = []
for i, key in enumerate(keys_A):
    if key in key_to_idx_B:
        matched_idx_A.append(i)
        matched_idx_B.append(key_to_idx_B[key])

matched_idx_A = np.array(matched_idx_A)
matched_idx_B = np.array(matched_idx_B)

print(f"\nMatched donuts: {len(matched_idx_A)}")
print(f"  Unmatched in A: {len(tA) - len(matched_idx_A)}")
print(f"  Unmatched in B: {len(tB) - len(matched_idx_B)}")

# Extract matched Zernike arrays
zk_A = np.stack(tA[zk_col])[matched_idx_A]
zk_B = np.stack(tB[zk_col])[matched_idx_B]
print(f"\nMatched Zernike arrays: A={zk_A.shape}, B={zk_B.shape}")

# Also keep matched metadata for reference
matched_day_obs = np.array(tA['day_obs'])[matched_idx_A]
matched_seq_num = np.array(tA['seq_num'])[matched_idx_A]
matched_thx = np.array(tA[thx_col])[matched_idx_A]
matched_thy = np.array(tA[thy_col])[matched_idx_A]

n_images = len(set(zip(matched_day_obs.tolist(), matched_seq_num.tolist())))
print(f"  Spanning {n_images} unique images")

<a id='scatter'></a>
## Hexbin Density Plots

Hexbin density plots of Zernike values (Algorithm A vs Algorithm B) for all matched donuts.
Axis ranges cover the 2nd to 98th percentile across both algorithms. Units are microns.

In [ ]:
# Hexbin density plots: all common Zernike coefficients
n_zk = len(iZs_common)
n_cols = 4
n_rows = int(np.ceil(n_zk / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4.5 * n_rows))
axes = np.atleast_2d(axes)

for ax_idx, iZ in enumerate(iZs_common):
    ax = axes[ax_idx // n_cols, ax_idx % n_cols]
    
    vals_A = zk_A[:, iZidx_A[iZ]]
    vals_B = zk_B[:, iZidx_B[iZ]]
    
    # Range covers 2%-98% of points in each algorithm
    lo = min(np.percentile(vals_A, 2), np.percentile(vals_B, 2))
    hi = max(np.percentile(vals_A, 98), np.percentile(vals_B, 98))
    
    # Hexbin density plot
    hb = ax.hexbin(vals_A, vals_B, gridsize=60, cmap='viridis',
                   mincnt=1, extent=[lo, hi, lo, hi],
                   bins='log', rasterized=True)
    fig.colorbar(hb, ax=ax, label='log10(N)')
    
    # 1:1 line
    ax.plot([lo, hi], [lo, hi], 'r-', linewidth=1.0, alpha=0.8)
    
    # Stats
    diff = vals_B - vals_A
    mean_diff = np.mean(diff)
    std_diff = np.std(diff)
    ax.text(0.03, 0.97, f'\u0394={mean_diff:.3f}\n\u03c3={std_diff:.3f} \u03bcm',
            transform=ax.transAxes, ha='left', va='top', fontsize=8,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    ax.set_title(f'Z{iZ}')
    ax.set_xlabel(f'{label_A} [\u03bcm]')
    ax.set_ylabel(f'{label_B} [\u03bcm]')
    ax.set_aspect('equal')
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)

# Hide unused axes
for ax_idx in range(n_zk, n_rows * n_cols):
    axes[ax_idx // n_cols, ax_idx % n_cols].set_visible(False)

fig.suptitle(f'Zernike Comparison: {label_A} vs {label_B} ({coord_sys}, {day_obs_min}-{day_obs_max})',
             fontsize=14, y=1.01)
plt.tight_layout()

output_file = f'{output_dir}/donutalgo_hexbin_{prefix_A}_vs_{prefix_B}_{day_obs_min}_{day_obs_max}.pdf'
fig.savefig(output_file, dpi=150, bbox_inches='tight')
print(f"Saved: {output_file}")
plt.show()

In [ ]:
# Difference histograms (B - A) for all common Zernikes
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3.5 * n_rows))
axes = np.atleast_2d(axes)

for ax_idx, iZ in enumerate(iZs_common):
    ax = axes[ax_idx // n_cols, ax_idx % n_cols]
    
    diff = zk_B[:, iZidx_B[iZ]] - zk_A[:, iZidx_A[iZ]]
    plo, phi = np.percentile(diff, [1, 99])
    
    ax.hist(diff, bins=100, range=(plo, phi), log=True,
            edgecolor='black', linewidth=0.3, alpha=0.7)
    ax.axvline(0, color='red', linewidth=0.8, linestyle='--')
    ax.set_title(f'Z{iZ}')
    ax.set_xlabel(f'\u0394 [\u03bcm]')
    ax.set_ylabel('Count')
    
    mean_d = np.mean(diff)
    std_d = np.std(diff)
    ax.text(0.97, 0.95, f'\u0394={mean_d:.3f}\n\u03c3={std_d:.3f} \u03bcm',
            transform=ax.transAxes, ha='right', va='top', fontsize=8,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

for ax_idx in range(n_zk, n_rows * n_cols):
    axes[ax_idx // n_cols, ax_idx % n_cols].set_visible(False)

fig.suptitle(f'Zernike Differences: {label_B} - {label_A} ({coord_sys}, {day_obs_min}-{day_obs_max})',
             fontsize=14, y=1.01)
plt.tight_layout()

output_file = f'{output_dir}/donutalgo_diff_{prefix_A}_vs_{prefix_B}_{day_obs_min}_{day_obs_max}.pdf'
fig.savefig(output_file, dpi=150, bbox_inches='tight')
print(f"Saved: {output_file}")
plt.show()